# Komentarų agresijos filtras – treniravimas Google Colab

**Prieš pradedant:** *Runtime → Change runtime type → T4 GPU*.

Užrašai paleidžia visą eksperimentą: duomenų paruošimas → EDA → Baseline → DistilBERT → palyginimas → XAI.

In [ ]:
!nvidia-smi -L

## 1. Kodas
Atsisiunčiamas grupės kodas iš GitHub.

In [ ]:
%cd /content
!rm -rf Di-grupinis-komanda-Y
!git clone https://github.com/neiluciokas123-debug/Di-grupinis-komanda-Y.git
%cd /content/Di-grupinis-komanda-Y
!pip install -q lime tabulate
!ls

## 2. Duomenys
Įkelkite `train.csv` **arba** visą Kaggle archyvą `jigsaw-toxic-comment-classification-challenge.zip` (atsisiųstą iš konkurso Data skirtuko).

In [ ]:
import os, glob, zipfile
from google.colab import files
%cd /content/Di-grupinis-komanda-Y
os.makedirs('data/raw', exist_ok=True)

up = files.upload()  # train.csv ARBA Kaggle zip archyvas
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('data/raw')
    elif name.endswith('.csv') and 'train' in name:
        os.replace(name, 'data/raw/train.csv')

for z in glob.glob('data/raw/*.zip'):  # Kaggle archyve yra train.csv.zip
    zipfile.ZipFile(z).extractall('data/raw')

!ls -lh data/raw

## 3. Paruošimas ir EDA

In [ ]:
!python -m src.data
!python -m src.eda

## 4. Bazinis eksperimentas (TF-IDF + LogisticRegression)
Trunka ~5–10 min (CPU).

In [ ]:
!python -m src.train_baseline

## 5. Patobulintas eksperimentas (DistilBERT)
50 000 komentarų × 2 epochos ≈ 15–25 min T4 GPU. Visam rinkiniui: `--max-train 0` (~1 val.).

In [ ]:
!python -m src.train_distilbert --max-train 50000 --epochs 2

## 6. Palyginimas ir paaiškinamumas

In [ ]:
!python -m src.evaluate
!python -m src.explain

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open('reports/metrics/comparison.md').read()))
for f in ['metrics_comparison','confusion_matrices','roc_pr_curves','categories_f1','distilbert_training','bias_test','eda_labels','eda_toxic_words','xai_baseline_global','xai_occlusion_0']:
    p = f'reports/figures/{f}.png'
    if os.path.exists(p):
        display(Markdown(f'### {f}')); display(Image(p, width=700))

## 7. Išsaugoti rezultatus
Atsisiųskite modelius ir ataskaitos grafikus (modelių į GitHub nekelkite – per dideli).

In [ ]:
!zip -q -r rezultatai.zip models reports
files.download('rezultatai.zip')